In [1]:
import pandas as pd

columns = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent',
    'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count',
    'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

DATA_PATH = "KDDTrain+.csv"
df = pd.read_csv(DATA_PATH, header=None, names=columns)

# 2. Drop metadata
df.drop(columns=['difficulty_level'], inplace=True)

# 3. Binary target
df['target'] = df['label'].apply(lambda x: 0 if x == 'normal' else 1)
df.drop(columns=['label'], inplace=True)

# 4. Encode categoricals - converting categorical text columns into machine-readable numeric form.

# One-hot encoding
df = pd.get_dummies(df, columns=['protocol_type', 'service', 'flag'])

# 5. Separate features and target
X = df.drop(columns=['target'])
y = df['target']

print("Shape:", df.shape)
print("Object cols left:", list(X.select_dtypes(include='object').columns))
print("Target balance:\n", y.value_counts())
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Shape: (125973, 123)
Object cols left: []
Target balance:
 target
0    67343
1    58630
Name: count, dtype: int64
Missing values: 0
Duplicate rows: 9


In [2]:
print(X.shape)
print(y.value_counts(normalize=True) * 100)

(125973, 122)
target
0    53.458281
1    46.541719
Name: proportion, dtype: float64


# Train Decision Tree

In [3]:
# Import necessary modules
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [4]:
# step 1 - Split the data into train and test

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(100778, 122) (25195, 122)


In [5]:
# Step 2: Create Decision Tree model
dt_model = DecisionTreeClassifier(random_state=42)

In [6]:
# Step 3: Train and Predict on test data

dt_model.fit(X_train, y_train)

#predict
y_pred_dt = dt_model.predict(X_test)

In [7]:
#Results

print("Test Accuracy:", accuracy_score(y_test, y_pred_dt))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("\nClassification Report:\n", classification_report(y_test, y_pred_dt))


Test Accuracy: 0.9984123834094066

Confusion Matrix:
 [[13445    24]
 [   16 11710]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     13469
           1       1.00      1.00      1.00     11726

    accuracy                           1.00     25195
   macro avg       1.00      1.00      1.00     25195
weighted avg       1.00      1.00      1.00     25195



In [8]:
# Check training accuracy
y_train_pred_dt = dt_model.predict(X_train)
print("Train Accuracy:", accuracy_score(y_train, y_train_pred_dt))

Train Accuracy: 0.9999503859969437
